[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Digital-AI-Finance/Introduction-to-Machine-Learning-notebooks/blob/master/block3.ipynb)

# Block 3: a small network, and a model that cheated

Nineteen minutes, together. Every answer is written. Read, run, look.

## 1. Images that ship with scikit-learn

1797 handwritten digits, each one eight pixels by eight. This is the data you
could not write a rule for in block 1.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits

digits = load_digits()
print("rows and columns:", digits.data.shape)

fig, ax = plt.subplots(1, 8, figsize=(9, 1.6))
for i in range(8):
    ax[i].imshow(digits.images[i], cmap="gray_r")
    ax[i].set_title(str(digits.target[i]))
    ax[i].axis("off")
plt.show()

## 2. One small network, trained

One hidden layer of sixteen neurons. Watch the number fall as it trains: that
is the loss, and training is the work of making it small. The loop stops on
its own once the loss has stopped falling.

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split

Xtr, Xte, ytr, yte = train_test_split(
    digits.data, digits.target, test_size=0.3, random_state=0)

net = MLPClassifier(hidden_layer_sizes=(16,), max_iter=1000,
                    random_state=0).fit(Xtr, ytr)

print("times round the loop:", net.n_iter_)
print("held back:", round(net.score(Xte, yte), 3))

plt.figure(figsize=(5, 2.5))
plt.plot(net.loss_curve_, color="#1e3a5f")
plt.title("the loss, falling")
plt.xlabel("how many times it went round the training loop")
plt.ylabel("the loss")
plt.show()

Nobody wrote a rule for any digit. Below are the held-back digits the network
got wrong, each titled with what it said and what the digit was.

In [ ]:
import numpy as np

said = net.predict(Xte)
wrong = np.flatnonzero(said != yte)
print("misread:", len(wrong), "of", len(yte))

rows = (len(wrong) + 8) // 9
fig, ax = plt.subplots(rows, 9, figsize=(9, 1.35 * rows))
for a in ax.flat:
    a.axis("off")
for a, i in zip(ax.flat, wrong):
    a.imshow(Xte[i].reshape(8, 8), cmap="gray_r")
    a.set_title("said %d, was %d" % (said[i], yte[i]), fontsize=8)
plt.tight_layout()
plt.show()

## 3. The same network, on sensor readings

Thirteen measurements from an instrument, running from below one to over a
thousand. A network adds its inputs together, so a column in the thousands
drowns the rest. `StandardScaler` puts every column on one scale first,
centred on zero with the same typical size, and it learns that scale from the
rows the network learns from. Then the same three lines as for the digits.

In [ ]:
from sklearn.datasets import load_wine
from sklearn.preprocessing import StandardScaler

wine = load_wine()
Xtr, Xte, ytr, yte = train_test_split(
    wine.data, wine.target, test_size=0.3, random_state=0)

scale = StandardScaler().fit(Xtr)
net2 = MLPClassifier(hidden_layer_sizes=(16,), max_iter=2000,
                     random_state=0).fit(scale.transform(Xtr), ytr)
print("held back:", round(net2.score(scale.transform(Xte), yte), 3))

## 4. Now a column that should not be there

A colleague sends you the same table with one extra column: the batch number
the sample was processed in. It looks harmless. Fit the same network.

In [ ]:
# The samples were processed one kind at a time, so the batch number happens
# to follow the answer. Nobody did this on purpose.
batch = wine.target * 100 + np.arange(len(wine.target)) % 7

leaky = np.column_stack([wine.data, batch])

Ltr, Lte, ytr, yte = train_test_split(
    leaky, wine.target, test_size=0.3, random_state=0)

scale3 = StandardScaler().fit(Ltr)
net3 = MLPClassifier(hidden_layer_sizes=(16,), max_iter=2000,
                     random_state=0).fit(scale3.transform(Ltr), ytr)
print("held back, with the extra column:",
      round(net3.score(scale3.transform(Lte), yte), 3))

## 5. What just happened

The held-back score hardly moved, and nothing warned you. The extra column
contains the answer: divide the batch number by a hundred and you have the
label. The held-back rows carry the same column, so a score on them cannot
tell a network that reads the measurements from one that reads the batch
number.

In [ ]:
print("batch numbers for each kind:")
for k in sorted(set(wine.target)):
    numbers = sorted(set(batch[wine.target == k]))[:4]
    print("  kind", k, "->", [int(b) for b in numbers], "...")

## 6. The next batch

After training, the laboratory keeps working. The next samples arrive with all
three kinds mixed together and go through as one new batch, numbered in the
three hundreds. Here are the held-back rows again, as that batch would deliver
them.

In [ ]:
fresh = Lte.copy()
fresh[:, -1] = 300 + np.arange(len(fresh)) % 7

print("new batch, without the extra column:",
      round(net2.score(scale.transform(Xte), yte), 3))
print("new batch, with the extra column:   ",
      round(net3.score(scale3.transform(fresh), yte), 3))

The network that never saw a batch number scores what it scored before. The
one that learned from the batch number falls, on the same rows, because the
column it leaned on no longer follows the answer.

## 7. Every column, scored on its own

One small decision tree per column, allowed two questions each, trained on the
rows the network learned from and scored on the rows held back.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

names = list(wine.feature_names) + ["batch_number"]
alone = []
for i, name in enumerate(names):
    small = DecisionTreeClassifier(max_depth=2, random_state=0)
    small.fit(Ltr[:, [i]], ytr)
    alone.append((round(small.score(Lte[:, [i]], yte), 3), name))

for score, name in sorted(alone, reverse=True)[:4]:
    print("%-16s %.3f" % (name, score))

This is data leakage. Two things give it away: a column that predicts the
answer on its own, and a score that falls when the model meets a new batch.
When one column is that good by itself, go and ask how it was filled in.